In [0]:
from pyspark.sql import functions as F

raw_path = "/Volumes/workforce_analytics/bronze/raw_data"

# Recursively find CSV files in the Volume
def list_csv_files(path):
    files = []

    for item in dbutils.fs.ls(path):
        if item.isDir():
            files.extend(list_csv_files(item.path))
        elif item.path.lower().endswith(".csv"):
            files.append(item.path)

    return files


csv_files = list_csv_files(raw_path)

for file_path in csv_files:
    print(file_path)

In [0]:
df=spark.read.csv("/Volumes/workforce_analytics/bronze/raw_data/source/dimensions/DimCancellationReason.csv",header=True,inferSchema=True)
df.display()

In [0]:
from pyspark.sql import functions as F

raw_path = "/Volumes/workforce_analytics/bronze/raw_data"

# Find CSV files recursively
def list_csv_files(path):
    files = []

    for item in dbutils.fs.ls(path):
        if item.isDir():
            files.extend(list_csv_files(item.path))
        elif item.path.lower().endswith(".csv"):
            files.append(item.path)

    return files


csv_files = list_csv_files(raw_path)

# Map Bronze table names to the actual filename prefixes
entities = {
    "fact_work": "FactWork_",
    "fact_shift": "FactShift_",
    "fact_shrinkage": "FactShrinkage_",
    "dim_county": "DimCounty",
    "dim_people": "DimPeople",
    "dim_shrinkage_type": "DimShrinkageType",
    "dim_cancellation_reason": "DimCancellationReason"
}

for table_name, filename_prefix in entities.items():

    matching_files = [
        path for path in csv_files
        if path.split("/")[-1].lower().startswith(filename_prefix.lower())
    ]

    print(f"\nLoading {table_name}")
    print(f"Files found: {len(matching_files)}")

    if not matching_files:
        print(f"No files found for {table_name}")
        continue

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(matching_files)
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_ingested_at", F.current_timestamp())
    )

    table_path = f"workforce_analytics.bronze.{table_name}_raw"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_path)
    )

    print(f"Loaded table: {table_path}")
    print(f"Rows loaded: {df.count()}")

In [0]:
spark.read.table("workforce_analytics.bronze.dim_county_raw").display()

In [0]:
from pyspark.sql import functions as F

for table_name in [
    "fact_work",
    "fact_shift",
    "fact_shrinkage"
]:
    table = f"workforce_analytics.bronze.{table_name}_raw"

    print(f"\n{table}")

    (
        spark.table(table)
        .groupBy("_source_file")
        .count()
        .orderBy("_source_file")
        .show(truncate=False)
    )

In [0]:
for table_name in [
    "fact_work",
    "fact_shift",
    "fact_shrinkage",
    "dim_county",
    "dim_people",
    "dim_shrinkage_type",
    "dim_cancellation_reason"
]:
    table = f"workforce_analytics.bronze.{table_name}_raw"

    print(f"\n--- {table} ---")
    spark.table(table).printSchema()

In [0]:
(
    spark.table("workforce_analytics.bronze.fact_work_raw")
    .groupBy("_source_file")
    .count()
    .orderBy("_source_file")
    .show(truncate=False)
)

### Inspection

In [0]:
from pyspark.sql import functions as F

base_path = "/Volumes/workforce_analytics/bronze/raw_data/source/facts"

fact_folders = [
    "fact_work",
    "fact_shift",
    "fact_shrinkage"
]

for folder in fact_folders:

    print(f"\n========== {folder} ==========")

    folder_path = f"{base_path}/{folder}"

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(folder_path)
    )

    print("Columns:")
    print(df.columns)

    print("Total rows across files:", df.count())

    print("Sample records:")
    display(df.limit(10))

In [0]:
from pyspark.sql import functions as F

tables = [
    "fact_work_raw",
    "fact_shift_raw",
    "fact_shrinkage_raw"
]

for table in tables:
    print(f"\n========== {table} ==========")

    df = spark.table(f"workforce_analytics.bronze.{table}")

    # Row count for each source file
    display(
        df.groupBy("_source_file")
          .count()
          .orderBy("_source_file")
    )

    # Show a few rows from each source file
    for source_file in [
        row["_source_file"]
        for row in df.select("_source_file").distinct().collect()
    ]:
        print(f"\nFile: {source_file}")
        display(
            df.filter(F.col("_source_file") == source_file)
              .drop("_ingested_at")
              .limit(5)
        )

In [0]:
from pyspark.sql import functions as F

tables_and_keys = {
    "fact_work_raw": "work_id",
    "fact_shift_raw": "shift_id",
    "fact_shrinkage_raw": "shrinkage_id"
}

for table, key in tables_and_keys.items():

    print(f"\n========== {table} ==========")

    df = spark.table(f"workforce_analytics.bronze.{table}")

    files = [
        row["_source_file"]
        for row in (
            df.select("_source_file")
              .distinct()
              .collect()
        )
    ]

    files = sorted(files)

    previous_ids = None

    for file in files:

        current = (
            df.filter(F.col("_source_file") == file)
              .select(key)
              .dropDuplicates()
        )

        duplicate_count = (
            df.filter(F.col("_source_file") == file)
              .count() - current.count()
        )

        if previous_ids is None:
            new_count = current.count()
            existing_count = 0
        else:
            new_count = current.join(
                previous_ids, on=key, how="left_anti"
            ).count()

            existing_count = current.count() - new_count

        print(file.split("/")[-1])
        print("Distinct IDs:", current.count())
        print("Duplicate rows within file:", duplicate_count)
        print("New IDs:", new_count)
        print("IDs already seen in earlier files:", existing_count)

        if previous_ids is None:
            previous_ids = current
        else:
            previous_ids = previous_ids.union(current).dropDuplicates()

In [0]:
from pyspark.sql import functions as F

tables_and_keys = {
    "fact_work_raw": "work_id",
    "fact_shift_raw": "shift_id",
    "fact_shrinkage_raw": "shrinkage_id"
}

for table, key in tables_and_keys.items():

    print(f"\n========== {table} ==========")

    df = spark.table(f"workforce_analytics.bronze.{table}")

    data_columns = [
        c for c in df.columns
        if c not in ["_source_file", "_ingested_at"]
    ]

    files = sorted([
        row["_source_file"]
        for row in df.select("_source_file").distinct().collect()
    ])

    # Start with the initial full load
    first_file = files[0]

    initial = (
        df.filter(F.col("_source_file") == first_file)
          .select(*data_columns)
          .dropDuplicates()
    )

    # Compare each subsequent file with the latest known records
    for file in files[1:]:

        delta = (
            df.filter(F.col("_source_file") == file)
              .select(*data_columns)
              .dropDuplicates()
        )

        comparison = (
            delta.alias("d")
            .join(
                initial.alias("p"),
                on=key,
                how="left"
            )
        )

        comparison = comparison.withColumn(
            "change_type",
            F.when(F.col(f"p.{key}").isNull(), "NEW")
             .when(
                 F.to_json(F.struct(*[
                     F.col(f"d.`{c}`").alias(c)
                     for c in data_columns
                 ])) ==
                 F.to_json(F.struct(*[
                     F.col(f"p.`{c}`").alias(c)
                     for c in data_columns
                 ])),
                 "UNCHANGED"
             )
             .otherwise("CHANGED")
        )

        print(f"\nFile: {file.split('/')[-1]}")
        display(
            comparison.groupBy("change_type").count()
        )

        print("Examples of changed records:")
        display(
            comparison.filter(F.col("change_type") == "CHANGED")
                      .select(
                          *[F.col(f"d.`{c}`").alias(c) for c in data_columns],
                          *[
                              F.col(f"p.`{c}`").alias(f"previous_{c}")
                              for c in data_columns
                              if c != key
                          ]
                      )
                      .limit(5)
        )

        # Update the accumulated state for the next comparison
        initial = (
            initial.alias("p")
            .join(
                delta.alias("d"),
                on=key,
                how="full"
            )
            .select(*[
                F.coalesce(F.col(f"d.`{c}`"), F.col(f"p.`{c}`")).alias(c)
                for c in data_columns
            ])
        )

    # Inspect duplicate IDs in the first file
    print("\nDuplicate IDs in initial file:")
    display(
        df.filter(F.col("_source_file") == first_file)
          .groupBy(key)
          .count()
          .filter(F.col("count") > 1)
    )

In [0]:
from pyspark.sql import functions as F

tables_and_keys = {
    "fact_work_raw": "work_id",
    "fact_shift_raw": "shift_id",
    "fact_shrinkage_raw": "shrinkage_id"
}

for table, key in tables_and_keys.items():

    print(f"\n========== {table} ==========")

    df = spark.table(f"workforce_analytics.bronze.{table}")

    first_file = (
        df.select("_source_file")
          .distinct()
          .orderBy("_source_file")
          .first()["_source_file"]
    )

    initial = df.filter(F.col("_source_file") == first_file)

    duplicate_ids = (
        initial.groupBy(key)
        .count()
        .filter(F.col("count") > 1)
        .select(key)
    )

    duplicate_rows = initial.join(
        duplicate_ids,
        on=key,
        how="inner"
    )

    data_columns = [
        c for c in df.columns
        if c not in ["_source_file", "_ingested_at"]
    ]

    print("Duplicate rows:")
    display(
        duplicate_rows
        .select(*data_columns)
        .orderBy(key)
    )

    print("Number of distinct versions per duplicate ID:")
    display(
        duplicate_rows
        .groupBy(key)
        .agg(
            F.count("*").alias("row_count"),
            F.countDistinct(
                F.to_json(F.struct(*data_columns))
            ).alias("distinct_versions")
        )
        .orderBy(key)
    )

In [0]:
from pyspark.sql import functions as F

checks = [
    ("fact_shift_raw", "shift_id"),
    ("fact_shrinkage_raw", "shrinkage_id")
]

for table, key in checks:

    df = spark.table(f"workforce_analytics.bronze.{table}")

    first_file = (
        df.select("_source_file")
          .distinct()
          .orderBy("_source_file")
          .first()["_source_file"]
    )

    initial = df.filter(F.col("_source_file") == first_file)

    columns = [
        c for c in df.columns
        if c not in ["_source_file", "_ingested_at"]
    ]

    display(
        initial.groupBy(key)
        .count()
        .filter(F.col("count") > 1)
        .join(
            initial.groupBy(key).agg(
                F.countDistinct(
                    F.to_json(F.struct(*columns))
                ).alias("distinct_versions")
            ),
            on=key
        )
        .orderBy(key)
    )